In [ ]:
# import sys
#! {sys.executable} -m pip install jupyter-black
import jupyter_black

jupyter_black.load()

In [ ]:
%%html
<!-- Define CSS styles for the notebook. -->
<style>
    /* Turn off margin and padding for Jupyter cells
     * since the HTML export will be one continuous report document. */
    .jp-Cell {
        padding: 0 !important;
    }
    .jp-Cell-outputWrapper {
        margin: 0 !important;
    }

    /* Make the empty output cells not take up space in the exported HTML */
    .jp-mod-noOutputs {padding: 0;}

    /* Set up the floating dropdown menu. */
    .menu-container {
        position: fixed;
        top: 20px;
        left: 20px;
        z-index: 9999;
    }
    .dropdown-menu {
        margin: 2px;
        margin-top: -2px;
        background-color: #F0F0F0;
    }

    div#rendered_cells {
        padding-top: 70px;
    }

    /* Make sure tab titles are the length of their text. */
    .lm-TabBar-tab {flex-basis: auto !important;}
</style>

In [ ]:
# Install dependencies.
# Commented out since they only need to run once. Uncomment as needed.
# import sys
# ! {sys.executable} -m pip install duckdb
# ! {sys.executable} -m pip install xdmod-data[report] anywidget
# ! {sys.executable} -m pip install --force-reinstall --no-deps git+https://github.com/aaronweeden/xdmod-data.git@whoami

In [ ]:
# Import modules and set styles.
from calendar import monthrange
from datetime import date, datetime
import duckdb
from IPython.display import display, Markdown, HTML
import ipywidgets as widgets
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import sys
import threading
import time
from xdmod_data.warehouse import DataWarehouse
import xdmod_data.themes
from xdmod_data.report import header, footer, set_styles

pio.renderers.default = "plotly_mimetype+notebook"
pio.templates.default = "timeseries"


def exception_handler(exception_type, exception, traceback):
    print(exception, file=sys.stderr)


# get_ipython()._showtraceback = exception_handler
display(set_styles())

In [ ]:
# Define report date constants.
YEAR = 2026
MONTH = 6
MONTH_YEAR = datetime(YEAR, MONTH, 1).strftime("%B %Y")
START_DATE = date(YEAR, MONTH, 1).strftime("%Y-%m-%d")
END_DATE = date(YEAR, MONTH, monthrange(YEAR, MONTH)[1]).strftime("%Y-%m-%d")
YEAR_MONTH = f"{YEAR}-{MONTH:02d}"

In [ ]:
# Set up the floating dropdown menu.
def get_loading_gif_html_widget(id=None):
    id_attribute = "" if id is None else f'id="{id}"'
    return widgets.HTML(
        value=f'<img {id_attribute} alt="Loading" class="loading-gif" src="data:image/gif;base64,R0lGODlhIAAgAPMAAP///wAAAMbGxoSEhLa2tpqamjY2NlZWVtjY2OTk5Ly8vB4eHgQEBAAAAAAAAAAAACH/C05FVFNDQVBFMi4wAwEAAAAh/hpDcmVhdGVkIHdpdGggYWpheGxvYWQuaW5mbwAh+QQJCgAAACwAAAAAIAAgAAAE5xDISWlhperN52JLhSSdRgwVo1ICQZRUsiwHpTJT4iowNS8vyW2icCF6k8HMMBkCEDskxTBDAZwuAkkqIfxIQyhBQBFvAQSDITM5VDW6XNE4KagNh6Bgwe60smQUB3d4Rz1ZBApnFASDd0hihh12BkE9kjAJVlycXIg7CQIFA6SlnJ87paqbSKiKoqusnbMdmDC2tXQlkUhziYtyWTxIfy6BE8WJt5YJvpJivxNaGmLHT0VnOgSYf0dZXS7APdpB309RnHOG5gDqXGLDaC457D1zZ/V/nmOM82XiHRLYKhKP1oZmADdEAAAh+QQJCgAAACwAAAAAIAAgAAAE6hDISWlZpOrNp1lGNRSdRpDUolIGw5RUYhhHukqFu8DsrEyqnWThGvAmhVlteBvojpTDDBUEIFwMFBRAmBkSgOrBFZogCASwBDEY/CZSg7GSE0gSCjQBMVG023xWBhklAnoEdhQEfyNqMIcKjhRsjEdnezB+A4k8gTwJhFuiW4dokXiloUepBAp5qaKpp6+Ho7aWW54wl7obvEe0kRuoplCGepwSx2jJvqHEmGt6whJpGpfJCHmOoNHKaHx61WiSR92E4lbFoq+B6QDtuetcaBPnW6+O7wDHpIiK9SaVK5GgV543tzjgGcghAgAh+QQJCgAAACwAAAAAIAAgAAAE7hDISSkxpOrN5zFHNWRdhSiVoVLHspRUMoyUakyEe8PTPCATW9A14E0UvuAKMNAZKYUZCiBMuBakSQKG8G2FzUWox2AUtAQFcBKlVQoLgQReZhQlCIJesQXI5B0CBnUMOxMCenoCfTCEWBsJColTMANldx15BGs8B5wlCZ9Po6OJkwmRpnqkqnuSrayqfKmqpLajoiW5HJq7FL1Gr2mMMcKUMIiJgIemy7xZtJsTmsM4xHiKv5KMCXqfyUCJEonXPN2rAOIAmsfB3uPoAK++G+w48edZPK+M6hLJpQg484enXIdQFSS1u6UhksENEQAAIfkECQoAAAAsAAAAACAAIAAABOcQyEmpGKLqzWcZRVUQnZYg1aBSh2GUVEIQ2aQOE+G+cD4ntpWkZQj1JIiZIogDFFyHI0UxQwFugMSOFIPJftfVAEoZLBbcLEFhlQiqGp1Vd140AUklUN3eCA51C1EWMzMCezCBBmkxVIVHBWd3HHl9JQOIJSdSnJ0TDKChCwUJjoWMPaGqDKannasMo6WnM562R5YluZRwur0wpgqZE7NKUm+FNRPIhjBJxKZteWuIBMN4zRMIVIhffcgojwCF117i4nlLnY5ztRLsnOk+aV+oJY7V7m76PdkS4trKcdg0Zc0tTcKkRAAAIfkECQoAAAAsAAAAACAAIAAABO4QyEkpKqjqzScpRaVkXZWQEximw1BSCUEIlDohrft6cpKCk5xid5MNJTaAIkekKGQkWyKHkvhKsR7ARmitkAYDYRIbUQRQjWBwJRzChi9CRlBcY1UN4g0/VNB0AlcvcAYHRyZPdEQFYV8ccwR5HWxEJ02YmRMLnJ1xCYp0Y5idpQuhopmmC2KgojKasUQDk5BNAwwMOh2RtRq5uQuPZKGIJQIGwAwGf6I0JXMpC8C7kXWDBINFMxS4DKMAWVWAGYsAdNqW5uaRxkSKJOZKaU3tPOBZ4DuK2LATgJhkPJMgTwKCdFjyPHEnKxFCDhEAACH5BAkKAAAALAAAAAAgACAAAATzEMhJaVKp6s2nIkolIJ2WkBShpkVRWqqQrhLSEu9MZJKK9y1ZrqYK9WiClmvoUaF8gIQSNeF1Er4MNFn4SRSDARWroAIETg1iVwuHjYB1kYc1mwruwXKC9gmsJXliGxc+XiUCby9ydh1sOSdMkpMTBpaXBzsfhoc5l58Gm5yToAaZhaOUqjkDgCWNHAULCwOLaTmzswadEqggQwgHuQsHIoZCHQMMQgQGubVEcxOPFAcMDAYUA85eWARmfSRQCdcMe0zeP1AAygwLlJtPNAAL19DARdPzBOWSm1brJBi45soRAWQAAkrQIykShQ9wVhHCwCQCACH5BAkKAAAALAAAAAAgACAAAATrEMhJaVKp6s2nIkqFZF2VIBWhUsJaTokqUCoBq+E71SRQeyqUToLA7VxF0JDyIQh/MVVPMt1ECZlfcjZJ9mIKoaTl1MRIl5o4CUKXOwmyrCInCKqcWtvadL2SYhyASyNDJ0uIiRMDjI0Fd30/iI2UA5GSS5UDj2l6NoqgOgN4gksEBgYFf0FDqKgHnyZ9OX8HrgYHdHpcHQULXAS2qKpENRg7eAMLC7kTBaixUYFkKAzWAAnLC7FLVxLWDBLKCwaKTULgEwbLA4hJtOkSBNqITT3xEgfLpBtzE/jiuL04RGEBgwWhShRgQExHBAAh+QQJCgAAACwAAAAAIAAgAAAE7xDISWlSqerNpyJKhWRdlSAVoVLCWk6JKlAqAavhO9UkUHsqlE6CwO1cRdCQ8iEIfzFVTzLdRAmZX3I2SfZiCqGk5dTESJeaOAlClzsJsqwiJwiqnFrb2nS9kmIcgEsjQydLiIlHehhpejaIjzh9eomSjZR+ipslWIRLAgMDOR2DOqKogTB9pCUJBagDBXR6XB0EBkIIsaRsGGMMAxoDBgYHTKJiUYEGDAzHC9EACcUGkIgFzgwZ0QsSBcXHiQvOwgDdEwfFs0sDzt4S6BK4xYjkDOzn0unFeBzOBijIm1Dgmg5YFQwsCMjp1oJ8LyIAACH5BAkKAAAALAAAAAAgACAAAATwEMhJaVKp6s2nIkqFZF2VIBWhUsJaTokqUCoBq+E71SRQeyqUToLA7VxF0JDyIQh/MVVPMt1ECZlfcjZJ9mIKoaTl1MRIl5o4CUKXOwmyrCInCKqcWtvadL2SYhyASyNDJ0uIiUd6GGl6NoiPOH16iZKNlH6KmyWFOggHhEEvAwwMA0N9GBsEC6amhnVcEwavDAazGwIDaH1ipaYLBUTCGgQDA8NdHz0FpqgTBwsLqAbWAAnIA4FWKdMLGdYGEgraigbT0OITBcg5QwPT4xLrROZL6AuQAPUS7bxLpoWidY0JtxLHKhwwMJBTHgPKdEQAACH5BAkKAAAALAAAAAAgACAAAATrEMhJaVKp6s2nIkqFZF2VIBWhUsJaTokqUCoBq+E71SRQeyqUToLA7VxF0JDyIQh/MVVPMt1ECZlfcjZJ9mIKoaTl1MRIl5o4CUKXOwmyrCInCKqcWtvadL2SYhyASyNDJ0uIiUd6GAULDJCRiXo1CpGXDJOUjY+Yip9DhToJA4RBLwMLCwVDfRgbBAaqqoZ1XBMHswsHtxtFaH1iqaoGNgAIxRpbFAgfPQSqpbgGBqUD1wBXeCYp1AYZ19JJOYgH1KwA4UBvQwXUBxPqVD9L3sbp2BNk2xvvFPJd+MFCN6HAAIKgNggY0KtEBAAh+QQJCgAAACwAAAAAIAAgAAAE6BDISWlSqerNpyJKhWRdlSAVoVLCWk6JKlAqAavhO9UkUHsqlE6CwO1cRdCQ8iEIfzFVTzLdRAmZX3I2SfYIDMaAFdTESJeaEDAIMxYFqrOUaNW4E4ObYcCXaiBVEgULe0NJaxxtYksjh2NLkZISgDgJhHthkpU4mW6blRiYmZOlh4JWkDqILwUGBnE6TYEbCgevr0N1gH4At7gHiRpFaLNrrq8HNgAJA70AWxQIH1+vsYMDAzZQPC9VCNkDWUhGkuE5PxJNwiUK4UfLzOlD4WvzAHaoG9nxPi5d+jYUqfAhhykOFwJWiAAAIfkECQoAAAAsAAAAACAAIAAABPAQyElpUqnqzaciSoVkXVUMFaFSwlpOCcMYlErAavhOMnNLNo8KsZsMZItJEIDIFSkLGQoQTNhIsFehRww2CQLKF0tYGKYSg+ygsZIuNqJksKgbfgIGepNo2cIUB3V1B3IvNiBYNQaDSTtfhhx0CwVPI0UJe0+bm4g5VgcGoqOcnjmjqDSdnhgEoamcsZuXO1aWQy8KAwOAuTYYGwi7w5h+Kr0SJ8MFihpNbx+4Erq7BYBuzsdiH1jCAzoSfl0rVirNbRXlBBlLX+BP0XJLAPGzTkAuAOqb0WT5AH7OcdCm5B8TgRwSRKIHQtaLCwg1RAAAOwAAAAAAAAAAAA=="/>'
    )


def toggle_menu(b):
    visible_display = "inline-flex"
    dropdown_menu.layout.display = (
        "none" if dropdown_menu.layout.display == visible_display else visible_display
    )


hamburger_button = widgets.Button(
    tooltip="Menu",
    icon="bars",
    layout=widgets.Layout(width="40px", height="40px", position="fixed"),
    button_style="primary",
)
hamburger_button.add_class("hamburger-button")
hamburger_button.on_click(toggle_menu)
toc_widget_heading = widgets.HTML(value=f"<div><b>Table of Contents</b></div>")
TOC_WIDGET = get_loading_gif_html_widget("toc-loading-gif")
dropdown_menu = widgets.VBox(
    [toc_widget_heading, TOC_WIDGET],
    layout=widgets.Layout(
        display="none",
        border="1px solid gray",
        padding="5px",
    ),
)
dropdown_menu.add_class("dropdown-menu")
menu_container = widgets.VBox([hamburger_button, dropdown_menu])
menu_container.add_class("menu-container")
display(menu_container)
display(HTML("""
<script>
(function() {
    function moveMenu() {
        const menu = document.querySelector('.menu-container');
        if (!menu) {
            setTimeout(moveMenu, 100);
            return;
        }
        document.body.appendChild(menu);
    }
    moveMenu();
})();
</script>
"""))

In [ ]:
# import os
#
# os.environ["XDMOD_API_TOKEN"] = (
#    "7619.4d330f55af5ef252d566235ec982a86b6b9687162293f172868e279bd074d513"
# )

In [ ]:
# Get data about the current user.
dw = DataWarehouse("https://xdmod-dev.ccr.xdmod.org:9001")
# with dw:
#    USER_DATA = dw.whoami()
# USER_DATA['person_id'] = 341755
USER_DATA = {
    "person_id": 10332,
    "first_name": "Aaron",
    "last_name": "Weeden",
}
# Prepare to get data from the parquet files.
duckdb.query("PRAGMA disable_optimizer")
pass

In [ ]:
# Display header.
display(header())
display(
    Markdown(
        f"# ACCESS Metrics User Report for {USER_DATA['first_name']} {USER_DATA['last_name']} — {MONTH_YEAR}"
    )
)

In [ ]:
# Make sure there is data for the requested report.
DATA_DIRECTORY = f"/shared/access-user-report/{YEAR_MONTH}"

TICKET_INSTRUCTIONS = f'''If you have any questions, comments, or concerns, you can
[submit a ticket](https://support.access-ci.org/help-ticket) — choose Open a Ticket,
log in with your ACCESS account, and for "ACCESS User Support Issue" choose "XDMoD Question."'''


def exit():
    display(Markdown(TICKET_INSTRUCTIONS))
    display(footer())
    get_ipython().set_custom_exc((Exception,), lambda *args, **kwargs: None)
    raise Exception


if not Path(DATA_DIRECTORY).is_dir():
    display(Markdown("The requested report is not available. Please try again later."))
    exit()

In [ ]:
# Prepare to number sections, tables, and figures.
# SECTIONS variable below will be of the form:
# {
#     'Section 1 Title': {
#         'number': 1,
#         'subsections': {},
#     },
#     'Section 2 Title': {
#         'number': 2,
#         'subsections': {
#             'Section 2.1 Title': {
#                 'number': 1,
#                 'subsections': {
#                     'Section 2.1.1 Title': {
#                         'number': 1,
#                         'subsections': {},
#                     },
#                 },
#                 ...
#             },
#         },
#     },
#     ...
# }
SECTIONS = {}
# CURRENT_SECTION_TITLES below will contain the title of each current
# section/subsection/subsubsection, etc., e.g.,
# ['Section 1 Title', 'Section 1.1 Title', 'Section 1.1.1 Title']
CURRENT_SECTION_TITLES = []
CURRENT_SECTION_NUMBER_STR = ""
CURRENT_TABLE_NUMBER = 0
CURRENT_FIGURE_NUMBER = 0


def display_section_title(title, level):
    global SECTIONS
    global CURRENT_SECTION_TITLES
    global CURRENT_TABLE_NUMBER
    global CURRENT_FIGURE_NUMBER
    global CURRENT_SECTION_NUMBER_STR
    current_section_level = len(CURRENT_SECTION_TITLES)
    current_section = SECTIONS
    for i in range(0, current_section_level - 1):
        t = CURRENT_SECTION_TITLES[i]
        current_section = current_section[t]["subsections"]
    if level == current_section_level:
        add_section_at_same_level(title, level, current_section)
    elif level == current_section_level + 1:
        if level > 1:
            t = CURRENT_SECTION_TITLES[-1]
            current_section = current_section[t]["subsections"]
        CURRENT_SECTION_TITLES.append(title)
        current_section_level += 1
        current_section[title] = {
            "number": 1,
            "subsections": {},
        }
    elif level < current_section_level:
        while level < current_section_level:
            CURRENT_SECTION_TITLES.pop()
            current_section_level -= 1
        current_section = SECTIONS
        for i in range(0, current_section_level - 1):
            t = CURRENT_SECTION_TITLES[i]
            current_section = current_section[t]["subsections"]
        add_section_at_same_level(title, level, current_section)
    else:
        raise RuntimeError(
            f"Section title is at wrong level — jumped from level {current_section_level} to level {level}."
        )
    hashes = "#" + ("#" * level)
    current_section = SECTIONS
    section_numbers = []
    for i in range(0, current_section_level):
        t = CURRENT_SECTION_TITLES[i]
        current_section_number = current_section[t]["number"]
        section_numbers.append(current_section_number)
        current_section = current_section[t]["subsections"]
    CURRENT_SECTION_NUMBER_STR = ".".join([str(i) for i in section_numbers])
    display(Markdown(f"{hashes} {CURRENT_SECTION_NUMBER_STR}. {title}"))


def add_section_at_same_level(title, level, current_section):
    global CURRENT_SECTION_TITLES
    global CURRENT_TABLE_NUMBER
    global CURRENT_FIGURE_NUMBER
    if title not in current_section:
        t = CURRENT_SECTION_TITLES[-1]
        current_section_number = current_section[t]["number"]
        current_section[title] = {
            "number": current_section_number + 1,
            "subsections": {},
        }
        if level == 1:
            CURRENT_TABLE_NUMBER = 0
            CURRENT_FIGURE_NUMBER = 0
    CURRENT_SECTION_TITLES[-1] = title

In [ ]:
display_section_title("Introduction", level=1)
display(Markdown(f"""
This report summarizes usage and performance information for the jobs you ran on ACCESS-allocated resources
in {MONTH_YEAR}. It also provides information for other users on your ACCESS project(s), at your institution,
and in the same field of science as your project(s) so you can compare your usage and performance to theirs.

The metrics in this report use a unit of **ACCESS Credit Equivalents**, which is a normalized unit used to compare
usage across different types of resources with different compute capabilities. One ACCESS Credit Equivalent is defined
as one CPU Hour on SDSC Expanse (an AMD EPYC 7742 based compute resource).

The source of data for this report is [ACCESS XDMoD](https://xdmod.access-ci.org), which has data collected from ACCESS
Resource Providers. Please note the following caveats about the data in this report:

- ACCESS XDMoD is intended as a historical data source rather than a real-time data source, and there is a multi-day
  delay in data being available to view in ACCESS XDMoD.
- Data are occasionally missing or incorrect, and not every ACCESS-allocated resource provides all of the types of
  data that ACCESS XDMoD collects.
- Data are often backfilled and corrected by the ACCESS Metrics team, meaning the data in this report may change if
  loaded later, and data may be inconsistent with what is available in ACCESS XDMoD. ACCESS XDMoD should be treated
  as more up-to-date than this report except where otherwise noted.
- This report document will exist for a limited time (approximately three months), but the data will persist in
  ACCESS XDMoD.
"""))

In [ ]:
section_data = []


def display_section(title, level, render_function):
    display_section_title(title, level=level)
    container_widget = widgets.VBox([get_loading_gif_html_widget()])
    display(container_widget)
    section_data.append(
        {
            "container_widget": container_widget,
            "render_function": render_your_data_section,
        }
    )

In [ ]:
def render_your_data_section(container_widget):
    global my_projects_usage_df, my_projects_df
    my_projects_usage_df = duckdb.query(f"""
        SELECT *
        FROM '{DATA_DIRECTORY}/usage.parquet'
        WHERE person_id = {USER_DATA['person_id']} or pi_id = {USER_DATA['person_id']}
    """).to_df()
    my_projects_df = duckdb.query(f"""
        SELECT DISTINCT a.*
        FROM my_projects_usage_df
        JOIN '{DATA_DIRECTORY}/dimensions/allocation.parquet' AS a
        ON my_projects_usage_df.allocation_id = a.id
    """).to_df()
    # If the current user has no data, say so.
    if my_projects_df.empty:
        display(
            Markdown(
                f"This report has no usage data for your ACCESS projects for {MONTH_YEAR}."
            )
        )
        exit()
    output_widget = widgets.Output()
    print(my_projects_usage_df)
    with output_widget:
        display_timeseries_bar_plot()
    container_widget.children = [output_widget]


def display_timeseries_bar_plot():
    chart_config = {
        "data": [
            {
                "name": "ACCESS",
                "meta": {"primarySeries": True},
                "customdata": ["ACCESS"],
                "cliponaxis": False,
                "marker": {
                    "color": "#1199ff",
                    "line": {"width": 0, "color": "#0053b9"},
                    "opacity": 1,
                },
                "type": "bar",
                "yaxis": "y1",
                "showlegend": True,
                "hovertext": [
                    "0.0",
                    "81.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "136.3",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "20.1",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                    "0.0",
                ],
                "hovertemplate": "ACCESS: <b>%{y:,.1f}</b> <extra></extra>",
                "hoverlabel": {
                    "align": "left",
                    "bgcolor": "rgba(255, 255, 255, 0.8)",
                    "bordercolor": "#1199ff",
                    "font": {
                        "size": 12.8,
                        "color": "#000000",
                        "family": "Lucida Grande, Lucida Sans Unicode, Arial, Helvetica, sans-serif",
                    },
                    "namelength": -1,
                },
                "text": [],
                "textposition": "outside",
                "textangle": -90,
                "textfont": {
                    "size": 14,
                    "color": "#1199ff",
                    "family": "'Lucida Grande', 'Lucida Sans Unicode', Arial, Helvetica, sans-serif",
                },
                "x": [
                    1780286400000,
                    1780372800000,
                    1780459200000,
                    1780545600000,
                    1780632000000,
                    1780718400000,
                    1780804800000,
                    1780891200000,
                    1780977600000,
                    1781064000000,
                    1781150400000,
                    1781236800000,
                    1781323200000,
                    1781409600000,
                    1781496000000,
                    1781582400000,
                    1781668800000,
                    1781755200000,
                    1781841600000,
                    1781928000000,
                    1782014400000,
                    1782100800000,
                    1782187200000,
                    1782273600000,
                    1782360000000,
                    1782446400000,
                    1782532800000,
                    1782619200000,
                    1782705600000,
                    1782792000000,
                ],
                "y": [
                    None,
                    80.9782,
                    None,
                    None,
                    None,
                    None,
                    None,
                    136.253,
                    None,
                    None,
                    None,
                    None,
                    None,
                    None,
                    None,
                    20.0886,
                    None,
                    None,
                    None,
                    None,
                    None,
                    None,
                    None,
                    None,
                    None,
                    None,
                    None,
                    None,
                    None,
                    None,
                ],
                "offsetgroup": "group3",
                "legendgroup": 3,
                "legendrank": 3,
                "visible": True,
            }
        ],
        "layout": {
            "annotations": [
                {
                    "name": "title",
                    "text": "ACCESS Credit Equivalents Charged by Day",
                    "xref": "paper",
                    "yref": "paper",
                    "xanchor": "center",
                    "yanchor": "bottom",
                    "x": 0.5,
                    "y": 1.05,
                    "font": {
                        "color": "#000000",
                        "family": "Lucida Grande, Lucida Sans Unicode, Arial, Helvetica, sans-serif",
                        "size": 19,
                    },
                    "showarrow": False,
                    "captureevents": True,
                },
                {
                    "name": "subtitle",
                    "text": "User =  Weeden, Aaron Michael - SUNY Buffalo",
                    "xref": "paper",
                    "yref": "paper",
                    "xanchor": "center",
                    "yanchor": "bottom",
                    "x": 0.5,
                    "y": 1,
                    "showarrow": False,
                    "captureevents": True,
                    "font": {"color": "#5078a0", "size": 15},
                },
                {
                    "name": "credits",
                    "text": "2026-06-01 to 2026-06-30  Src: ACDB. Powered by XDMoD/Plotly JS",
                    "font": {
                        "color": "#909090",
                        "size": 9.6,
                        "family": "Lucida Grande, Lucida Sans Unicode, Arial, Helvetica, sans-serif",
                    },
                    "xref": "paper",
                    "yref": "paper",
                    "xanchor": "right",
                    "yanchor": "bottom",
                    "x": 1,
                    "y": 0,
                    "xshift": 25,
                    "showarrow": False,
                },
            ],
            "legend": {
                "itemwidth": 40,
                "itemsizing": "constant",
                "bgcolor": "#ffffff",
                "borderwidth": 0,
                "xref": "container",
                "yref": "container",
                "traceorder": "normal",
                "font": {
                    "family": "Lucida Grande, Lucida Sans Unicode, Arial, Helvetica, sans-serif",
                    "color": "#274b6d",
                    "size": 15,
                },
                "xanchor": "center",
                "yanchor": "bottom",
                "x": 0.5,
                "y": 0.02,
                "orientation": "h",
            },
            "hovermode": "x unified",
            "hoverdistance": 1,
            "hoverlabel": {
                "align": "left",
                "bgcolor": "rgba(255, 255, 255, 0.8)",
                "font": {
                    "size": 12.8,
                    "color": "#333333",
                    "family": "Lucida Grande, Lucida Sans Unicode, Arial, Helvetica, sans-serif",
                },
                "namelength": -1,
                "bordercolor": "#1199ff",
            },
            "barmode": "group",
            "images": [],
            "margin": {"t": 45, "r": 25, "b": 25, "l": 80},
            "yaxis": {
                "automargin": True,
                "autorangeoptions": {"minallowed": 0, "maxallowed": None},
                "layer": "below traces",
                "title": {
                    "text": "<b>ACCESS Credit Equivalents Charged: Total (SU)</b>",
                    "font": {
                        "color": "#1199ff",
                        "size": 15,
                        "family": "'Lucida Grande', 'Lucida Sans Unicode', Arial, Helvetica, sans-serif",
                    },
                },
                "exponentformat": "SI",
                "tickfont": {"size": 14, "color": "#606060"},
                "ticksuffix": " ",
                "tickprefix": None,
                "tickmode": "auto",
                "nticks": 10,
                "type": "linear",
                "rangemode": "tozero",
                "range": [0, None],
                "separatethousands": True,
                "overlaying": None,
                "linewidth": 2.75,
                "linecolor": "#c0d0e0",
                "side": "left",
                "anchor": "x",
                "autoshift": True,
                "gridwidth": 1.375,
                "gridcolor": "#c0c0c0",
                "zeroline": False,
            },
            "xaxis": {
                "automargin": True,
                "layer": "below traces",
                "title": {
                    "text": "<b></b>",
                    "font": {
                        "color": "#000000",
                        "size": 15,
                        "family": "'Lucida Grande', 'Lucida Sans Unicode', Arial, Helvetica, sans-serif",
                    },
                },
                "tickfont": {"size": 14, "color": "#606060"},
                "ticksuffix": " ",
                "tickformat": "%Y-%m-%d",
                "type": "date",
                "rangemode": "tozero",
                "hoverformat": "%Y-%m-%d",
                "tickmode": "array",
                "nticks": 20,
                "spikedash": "solid",
                "spikethickness": 1,
                "spikecolor": "#C0C0C0",
                "linewidth": 2.75,
                "linecolor": "#c0d0e0",
                "showgrid": False,
                "gridcolor": "#c0c0c0",
                "zeroline": False,
                "tickvals": [
                    1780286400000,
                    1780459200000,
                    1780632000000,
                    1780804800000,
                    1780977600000,
                    1781150400000,
                    1781323200000,
                    1781496000000,
                    1781668800000,
                    1781841600000,
                    1782014400000,
                    1782187200000,
                    1782360000000,
                    1782532800000,
                    1782705600000,
                    1782792000000,
                ],
            },
            "width": 1474,
            "height": 495,
        },
    }
    plot = go.Figure(chart_config)
    display(plot)


display_section("Your Data", 1, render_your_data_section)
render_your_data_section(section_data[-1]["container_widget"])

In [ ]:
def render_your_compute_job_performance_section():
    print("render_your_compute_job_performance_section")


display_section(
    "Your Compute Job Performance", 1, render_your_compute_job_performance_section
)

In [ ]:
def render_your_wait_times_section():
    print("render_your_wait_times_section")


display_section("Your Wait Times", 1, render_your_wait_times_section)

In [ ]:
# Lazily render the report sections.
def worker():
    time.sleep(0.1)  # Give time for browser to connect.
    for d in section_data:
        d["render_function"](d["container_widget"])


threading.Thread(target=worker, daemon=True).start()

In [ ]:
# Display the data.
def display_projects_tab_widget():
    titles = []
    additional_widget_data = []
    render_functions = []
    for project in my_projects_df.itertuples():
        titles.append(project.label.split(" - ")[0])
        additional_widget_data.append(
            {
                "project_id": int(project.id),
                "project_title": project.label,
            }
        )
        render_functions.append(render_project_tab)
    add_tabs(titles, additional_widget_data, render_functions)


def add_tabs(titles, additional_widget_data, render_functions):
    tab_widget = widgets.Tab()
    children = []
    widget_data = []
    for i in range(0, len(titles)):
        output_widget = widgets.Output()
        children.append(output_widget)
        if i < len(additional_widget_data):
            a = additional_widget_data[i]
        else:
            a = {}
        widget_data.append(
            {
                "output_widget": output_widget,
                "is_rendered": False,
                "render_function": render_functions[i],
            }
            | a
        )
    tab_widget.children = children
    tab_widget.titles = titles
    tab_widget.data = widget_data
    tab_widget.observe(change_tab, names="selected_index")
    change_tab({"owner": tab_widget, "new": 0})
    display(tab_widget)


def change_tab(change_data):
    tab_data = getattr(change_data["owner"], "data")[change_data["new"]]
    if not tab_data["is_rendered"]:
        inner_output = widgets.Output()
        with inner_output:
            display_loading_gif()
        with tab_data["output_widget"]:
            display(inner_output)
        new_output = widgets.Output()
        with new_output:
            tab_data["render_function"](tab_data)
        inner_output.clear_output(wait=True)
        with inner_output:
            display(new_output)
        tab_data["is_rendered"] = True


def render_project_tab(data):
    display(Markdown(f"### {data['project_title']}"))
    titles = ["Usage", "Performance"]
    additional_widget_data = []
    for title in titles:
        additional_widget_data.append(
            {
                "project_id": data["project_id"],
                "project_title": data["project_title"],
            }
        )
    add_tabs(
        titles=titles,
        additional_widget_data=additional_widget_data,
        render_functions=[render_usage_tab, render_performance_tab],
    )


def render_usage_tab(data):
    project_usage_df = duckdb.query(f"""
        SELECT *
        FROM my_projects_usage_df
        WHERE allocation_id = {data['project_id']}
    """).to_df()
    titles = [
        "Your Usage",
        "Your Project's Usage",
        "Your Institution's Usage",
        "Your Field of Science's Usage",
    ]
    additional_widget_data = []
    for _ in titles:
        additional_widget_data.append(
            {
                "project_usage_df": project_usage_df,
                "project_title": data["project_title"],
            }
        )
    add_tabs(
        titles=titles,
        additional_widget_data=additional_widget_data,
        render_functions=[
            render_your_usage_tab,
            render_your_projects_usage_tab,
            render_your_institutions_usage_tab,
            render_your_field_of_sciences_usage_tab,
        ],
    )


def render_your_usage_tab(data):
    project_usage_df = data["project_usage_df"]
    your_usage_df = duckdb.query(f"""
        SELECT *
        FROM project_usage_df
        WHERE person_id = {USER_DATA['person_id']}
    """).to_df()
    if your_usage_df.empty:
        display(
            Markdown(
                f"No data. Either you ran no jobs yourself in {MONTH_YEAR} or data are missing from this report."
            )
        )


def render_your_projects_usage_tab(data):
    project_usage_df = data["project_usage_df"]
    df = duckdb.query(f"""
        SELECT person_id, person.label AS 'User', total_ace AS 'ACCESS Credit Equivalents'
        FROM (
            SELECT person_id, SUM(total_ace) AS total_ace
            FROM project_usage_df
            GROUP BY person_id
            ORDER BY total_ace DESC, person_id DESC
            LIMIT 5
        ) AS t1
        JOIN '{DATA_DIRECTORY}/dimensions/person.parquet' AS person
        ON person.id = t1.person_id
    """).to_df()
    top_text = ""
    if len(df) == 5:
        top_text = " (Top 5)"
    display_bar_plot(
        df,
        x="User",
        title=f"Usage for {MONTH_YEAR} by User{top_text}",
        subtitle=f"Project: {data['project_title']}",
    )


def display_bar_plot(
    df,
    x=None,
    title=None,
    subtitle=None,
    color=None,
    labels=None,
    category_orders=None,
    color_discrete_map=None,
    vertical_legend=False,
    yaxis_tickformat=",",
    before_show=None,
    caption="",
):
    plot = px.bar(
        df,
        x=x,
        y="ACCESS Credit Equivalents",
        title=title,
        subtitle=subtitle,
        color=color,
        labels=labels,
        category_orders=category_orders,
        color_discrete_map=color_discrete_map,
        height=500,
    )
    plot.update_traces(
        hovertemplate="%{y:,.1f} ACCESS Credit Equivalents",
    )
    plot.update_layout(
        yaxis_tickformat=yaxis_tickformat,
        legend_title_text="",
        hovermode="x unified",
        hoverlabel_namelength=-1,
        margin=dict(
            b=150,
        ),
    )
    if before_show is not None:
        before_show(plot)
    plot.show(
        config={
            #'showAxisRangeEntryBoxes': False,
            #'toImageButtonOptions': {'filename': filename},
        }
    )
    return plot


def render_your_institutions_usage_tab(data):
    render_top_projects_bar_plot(
        project_usage_df=data["project_usage_df"],
        dimension_query=f"""
            SELECT institution.id, institution.label
            FROM project_usage_df
            JOIN '{DATA_DIRECTORY}/dimensions/institution.parquet' AS institution
                ON institution.id = project_usage_df.institution_id
            WHERE person_id = {USER_DATA['person_id']}
            UNION
            SELECT pi_institution.id, pi_institution.label
            FROM project_usage_df
            JOIN '{DATA_DIRECTORY}/dimensions/pi_institution.parquet' AS pi_institution
                ON pi_institution.id = project_usage_df.pi_institution_id
            WHERE pi_id = {USER_DATA['person_id']}
            LIMIT 1
        """,
        dimension_id="institution_id",
        dimension_label="Institution",
    )


def render_top_projects_bar_plot(
    project_usage_df, dimension_query, dimension_id, dimension_label
):
    dimension = duckdb.query(dimension_query).fetchone()
    df = duckdb.query(f"""
        SELECT
            allocation_id,
            allocation.label AS 'Project',
            pi.label AS 'PI',
            total_ace AS 'ACCESS Credit Equivalents'
        FROM (
            SELECT allocation_id, pi_id, SUM(total_ace) AS total_ace
            FROM '{DATA_DIRECTORY}/usage.parquet'
            WHERE {dimension_id} = {dimension[0]}
            GROUP BY allocation_id, pi_id
            ORDER BY total_ace DESC, allocation_id DESC
            LIMIT 5
        ) AS t1
        JOIN '{DATA_DIRECTORY}/dimensions/allocation.parquet' AS allocation
            ON allocation.id = t1.allocation_id
        JOIN '{DATA_DIRECTORY}/dimensions/pi.parquet' AS pi
            ON pi.id = t1.pi_id
    """).to_df()
    top_text = ""
    if len(df) == 5:
        top_text = " (Top 5)"
    print(dimension)
    display_bar_plot(
        df,
        x="Project",
        title=f"Usage for {MONTH_YEAR} by Project{top_text}",
        subtitle=f"{dimension_label}: {dimension[1]}",
    )
    for row in df.itertuples():
        request_number, project_title = row.Project.split(" - ", maxsplit=1)
        display(Markdown(f"""
- [{request_number}](https://allocations.access-ci.org/current-projects?_requestNumber={request_number}) - {project_title}
    - PI: {row.PI}
        """))


def render_your_field_of_sciences_usage_tab(data):
    render_top_projects_bar_plot(
        project_usage_df=data["project_usage_df"],
        dimension_query=f"""
            SELECT fieldofscience.id, fieldofscience.label
            FROM project_usage_df
            JOIN '{DATA_DIRECTORY}/dimensions/fieldofscience.parquet' AS fieldofscience
                ON fieldofscience.id = project_usage_df.fieldofscience_id
            LIMIT 1
        """,
        dimension_id="fieldofscience_id",
        dimension_label="Field of Science",
    )


def render_performance_tab(data):
    print(data)
    project_performance_df = duckdb.query(f"""
        SELECT *
        FROM '{DATA_DIRECTORY}/performance/cpuuser.parquet'
        WHERE person_id = {USER_DATA['person_id']} OR pi_id = {USER_DATA['person_id']}
        AND allocation_id = {data['project_id']}
    """).to_df()
    titles_to_metrics = {
        "CPU Utilization": "cpuuser",
        "Memory Utilization": "max_mem",
        "GPU Utilization": "gpu_usage_bucketid",
        "Wall Time Accuracy": "wall_time_accuracy_bucketid",
    }
    additional_widget_data = []
    render_functions = []
    for _, metric in titles_to_metrics:
        additional_widget_data.append(
            {
                "project_usage_df": project_usage_df,
                "project_title": data["project_title"],
                "metric": metric,
            }
        )
        render_functions.append(render_performance_histogram_tab)
    add_tabs(
        titles=titles,
        additional_widget_data=additional_widget_data,
        render_functions=render_functions,
    )


def render_performance_histogram_tab(data):
    print(data["metric"])


def display_loading_gif():
    display(
        HTML(
            f'<img alt="Loading" src="data:image/gif;base64,R0lGODlhIAAgAPMAAP///wAAAMbGxoSEhLa2tpqamjY2NlZWVtjY2OTk5Ly8vB4eHgQEBAAAAAAAAAAAACH/C05FVFNDQVBFMi4wAwEAAAAh/hpDcmVhdGVkIHdpdGggYWpheGxvYWQuaW5mbwAh+QQJCgAAACwAAAAAIAAgAAAE5xDISWlhperN52JLhSSdRgwVo1ICQZRUsiwHpTJT4iowNS8vyW2icCF6k8HMMBkCEDskxTBDAZwuAkkqIfxIQyhBQBFvAQSDITM5VDW6XNE4KagNh6Bgwe60smQUB3d4Rz1ZBApnFASDd0hihh12BkE9kjAJVlycXIg7CQIFA6SlnJ87paqbSKiKoqusnbMdmDC2tXQlkUhziYtyWTxIfy6BE8WJt5YJvpJivxNaGmLHT0VnOgSYf0dZXS7APdpB309RnHOG5gDqXGLDaC457D1zZ/V/nmOM82XiHRLYKhKP1oZmADdEAAAh+QQJCgAAACwAAAAAIAAgAAAE6hDISWlZpOrNp1lGNRSdRpDUolIGw5RUYhhHukqFu8DsrEyqnWThGvAmhVlteBvojpTDDBUEIFwMFBRAmBkSgOrBFZogCASwBDEY/CZSg7GSE0gSCjQBMVG023xWBhklAnoEdhQEfyNqMIcKjhRsjEdnezB+A4k8gTwJhFuiW4dokXiloUepBAp5qaKpp6+Ho7aWW54wl7obvEe0kRuoplCGepwSx2jJvqHEmGt6whJpGpfJCHmOoNHKaHx61WiSR92E4lbFoq+B6QDtuetcaBPnW6+O7wDHpIiK9SaVK5GgV543tzjgGcghAgAh+QQJCgAAACwAAAAAIAAgAAAE7hDISSkxpOrN5zFHNWRdhSiVoVLHspRUMoyUakyEe8PTPCATW9A14E0UvuAKMNAZKYUZCiBMuBakSQKG8G2FzUWox2AUtAQFcBKlVQoLgQReZhQlCIJesQXI5B0CBnUMOxMCenoCfTCEWBsJColTMANldx15BGs8B5wlCZ9Po6OJkwmRpnqkqnuSrayqfKmqpLajoiW5HJq7FL1Gr2mMMcKUMIiJgIemy7xZtJsTmsM4xHiKv5KMCXqfyUCJEonXPN2rAOIAmsfB3uPoAK++G+w48edZPK+M6hLJpQg484enXIdQFSS1u6UhksENEQAAIfkECQoAAAAsAAAAACAAIAAABOcQyEmpGKLqzWcZRVUQnZYg1aBSh2GUVEIQ2aQOE+G+cD4ntpWkZQj1JIiZIogDFFyHI0UxQwFugMSOFIPJftfVAEoZLBbcLEFhlQiqGp1Vd140AUklUN3eCA51C1EWMzMCezCBBmkxVIVHBWd3HHl9JQOIJSdSnJ0TDKChCwUJjoWMPaGqDKannasMo6WnM562R5YluZRwur0wpgqZE7NKUm+FNRPIhjBJxKZteWuIBMN4zRMIVIhffcgojwCF117i4nlLnY5ztRLsnOk+aV+oJY7V7m76PdkS4trKcdg0Zc0tTcKkRAAAIfkECQoAAAAsAAAAACAAIAAABO4QyEkpKqjqzScpRaVkXZWQEximw1BSCUEIlDohrft6cpKCk5xid5MNJTaAIkekKGQkWyKHkvhKsR7ARmitkAYDYRIbUQRQjWBwJRzChi9CRlBcY1UN4g0/VNB0AlcvcAYHRyZPdEQFYV8ccwR5HWxEJ02YmRMLnJ1xCYp0Y5idpQuhopmmC2KgojKasUQDk5BNAwwMOh2RtRq5uQuPZKGIJQIGwAwGf6I0JXMpC8C7kXWDBINFMxS4DKMAWVWAGYsAdNqW5uaRxkSKJOZKaU3tPOBZ4DuK2LATgJhkPJMgTwKCdFjyPHEnKxFCDhEAACH5BAkKAAAALAAAAAAgACAAAATzEMhJaVKp6s2nIkolIJ2WkBShpkVRWqqQrhLSEu9MZJKK9y1ZrqYK9WiClmvoUaF8gIQSNeF1Er4MNFn4SRSDARWroAIETg1iVwuHjYB1kYc1mwruwXKC9gmsJXliGxc+XiUCby9ydh1sOSdMkpMTBpaXBzsfhoc5l58Gm5yToAaZhaOUqjkDgCWNHAULCwOLaTmzswadEqggQwgHuQsHIoZCHQMMQgQGubVEcxOPFAcMDAYUA85eWARmfSRQCdcMe0zeP1AAygwLlJtPNAAL19DARdPzBOWSm1brJBi45soRAWQAAkrQIykShQ9wVhHCwCQCACH5BAkKAAAALAAAAAAgACAAAATrEMhJaVKp6s2nIkqFZF2VIBWhUsJaTokqUCoBq+E71SRQeyqUToLA7VxF0JDyIQh/MVVPMt1ECZlfcjZJ9mIKoaTl1MRIl5o4CUKXOwmyrCInCKqcWtvadL2SYhyASyNDJ0uIiRMDjI0Fd30/iI2UA5GSS5UDj2l6NoqgOgN4gksEBgYFf0FDqKgHnyZ9OX8HrgYHdHpcHQULXAS2qKpENRg7eAMLC7kTBaixUYFkKAzWAAnLC7FLVxLWDBLKCwaKTULgEwbLA4hJtOkSBNqITT3xEgfLpBtzE/jiuL04RGEBgwWhShRgQExHBAAh+QQJCgAAACwAAAAAIAAgAAAE7xDISWlSqerNpyJKhWRdlSAVoVLCWk6JKlAqAavhO9UkUHsqlE6CwO1cRdCQ8iEIfzFVTzLdRAmZX3I2SfZiCqGk5dTESJeaOAlClzsJsqwiJwiqnFrb2nS9kmIcgEsjQydLiIlHehhpejaIjzh9eomSjZR+ipslWIRLAgMDOR2DOqKogTB9pCUJBagDBXR6XB0EBkIIsaRsGGMMAxoDBgYHTKJiUYEGDAzHC9EACcUGkIgFzgwZ0QsSBcXHiQvOwgDdEwfFs0sDzt4S6BK4xYjkDOzn0unFeBzOBijIm1Dgmg5YFQwsCMjp1oJ8LyIAACH5BAkKAAAALAAAAAAgACAAAATwEMhJaVKp6s2nIkqFZF2VIBWhUsJaTokqUCoBq+E71SRQeyqUToLA7VxF0JDyIQh/MVVPMt1ECZlfcjZJ9mIKoaTl1MRIl5o4CUKXOwmyrCInCKqcWtvadL2SYhyASyNDJ0uIiUd6GGl6NoiPOH16iZKNlH6KmyWFOggHhEEvAwwMA0N9GBsEC6amhnVcEwavDAazGwIDaH1ipaYLBUTCGgQDA8NdHz0FpqgTBwsLqAbWAAnIA4FWKdMLGdYGEgraigbT0OITBcg5QwPT4xLrROZL6AuQAPUS7bxLpoWidY0JtxLHKhwwMJBTHgPKdEQAACH5BAkKAAAALAAAAAAgACAAAATrEMhJaVKp6s2nIkqFZF2VIBWhUsJaTokqUCoBq+E71SRQeyqUToLA7VxF0JDyIQh/MVVPMt1ECZlfcjZJ9mIKoaTl1MRIl5o4CUKXOwmyrCInCKqcWtvadL2SYhyASyNDJ0uIiUd6GAULDJCRiXo1CpGXDJOUjY+Yip9DhToJA4RBLwMLCwVDfRgbBAaqqoZ1XBMHswsHtxtFaH1iqaoGNgAIxRpbFAgfPQSqpbgGBqUD1wBXeCYp1AYZ19JJOYgH1KwA4UBvQwXUBxPqVD9L3sbp2BNk2xvvFPJd+MFCN6HAAIKgNggY0KtEBAAh+QQJCgAAACwAAAAAIAAgAAAE6BDISWlSqerNpyJKhWRdlSAVoVLCWk6JKlAqAavhO9UkUHsqlE6CwO1cRdCQ8iEIfzFVTzLdRAmZX3I2SfYIDMaAFdTESJeaEDAIMxYFqrOUaNW4E4ObYcCXaiBVEgULe0NJaxxtYksjh2NLkZISgDgJhHthkpU4mW6blRiYmZOlh4JWkDqILwUGBnE6TYEbCgevr0N1gH4At7gHiRpFaLNrrq8HNgAJA70AWxQIH1+vsYMDAzZQPC9VCNkDWUhGkuE5PxJNwiUK4UfLzOlD4WvzAHaoG9nxPi5d+jYUqfAhhykOFwJWiAAAIfkECQoAAAAsAAAAACAAIAAABPAQyElpUqnqzaciSoVkXVUMFaFSwlpOCcMYlErAavhOMnNLNo8KsZsMZItJEIDIFSkLGQoQTNhIsFehRww2CQLKF0tYGKYSg+ygsZIuNqJksKgbfgIGepNo2cIUB3V1B3IvNiBYNQaDSTtfhhx0CwVPI0UJe0+bm4g5VgcGoqOcnjmjqDSdnhgEoamcsZuXO1aWQy8KAwOAuTYYGwi7w5h+Kr0SJ8MFihpNbx+4Erq7BYBuzsdiH1jCAzoSfl0rVirNbRXlBBlLX+BP0XJLAPGzTkAuAOqb0WT5AH7OcdCm5B8TgRwSRKIHQtaLCwg1RAAAOwAAAAAAAAAAAA=="/>'
        )
    )


display_projects_tab_widget()

In [ ]:
# Display the footer and finish rendering the document.
exit()

In [ ]:
print(duckdb.query('''
SELECT allocation_id, pi_id, SUM(total_ace) AS total_ace
    FROM '/shared/access-user-report/2026-06/usage.parquet'
    WHERE fieldofscience_id = 218
    GROUP BY allocation_id, pi_id
    ORDER BY total_ace DESC, allocation_id DESC
    LIMIT 5
''').to_df())

In [ ]:
print(duckdb.query('''
SELECT person.label, total_ace
FROM (
    SELECT person_id, resource_id, SUM(total_ace) AS total_ace
    FROM '/shared/access-user-report/2026-06/usage.parquet'
    WHERE allocation_id = 33316
    GROUP BY person_id, resource_id
    ORDER BY total_ace DESC, person_id, resource_id
) AS usage
JOIN '/shared/access-user-report/2026-06/dimensions/person.parquet' AS person ON person.id = usage.person_id
JOIN '/shared/access-user-report/2026-06/dimensions/resource.parquet' AS resource ON resource.id = usage.resource_id
''').to_df())

In [ ]:
print(duckdb.query('''
SELECT label, total_ace
FROM (
    SELECT person_id, sum(total_ace) AS total_ace
    FROM '/shared/access-user-report/2026-06/usage.parquet'
    WHERE allocation_id = 33316
    AND day_id = 202600152
    GROUP BY person_id
    ORDER BY total_ace DESC
) AS usage
JOIN '/shared/access-user-report/2026-06/dimensions/person.parquet' AS person ON person.id = usage.person_id
--JOIN '/shared/access-user-report/2026-06/dimensions/resource.parquet' AS resource ON resource.id = usage.resource_id
''').to_df())

In [ ]:
# 172962
# 351425
print(duckdb.query('''
SELECT person_id, SUM(total_ace) AS total_ace
FROM '/shared/access-user-report/2026-06/usage.parquet'
WHERE allocation_id = 33316
AND day_id = 202600152
GROUP BY person_id
ORDER BY total_ace DESC
''').to_df())

In [ ]:
print(duckdb.query('''
SELECT *
FROM '/shared/access-user-report/2026-06/usage.parquet'
WHERE person_id = 172962
''').to_df())

In [ ]:
print(duckdb.query('''
SELECT COUNT(*)
FROM '/shared/access-user-report/2026-06/usage.parquet'
''').to_df())

In [ ]:
Tabs = {
    'projects': {
        'tab_widget': widgets.Tab(),
    },
}

def build_projects_tab_widget():
    projects_tab_widget = Tabs['projects']['tab_widget']
    projects_tab_children = []
    projects_tab_titles = []
    for project_id, charge_number_and_title in my_project_charge_numbers_and_titles.items():
        if len(charge_number_and_title) == 1:
            charge_number = title = charge_number_and_title[0]
        else:
            (charge_number, title) = charge_number_and_title
        output = widgets.Output()
        projects_tab_children.append(output)
        projects_tab_titles.append(charge_number)
        project_tab_widget = widgets.Tab()
        project_tab_widget_titles = [
            'My Usage',
            'Total Project Usage',
            'Project Usage By User',
        ]
        project_tab_widget.children = [
            build_my_usage_tab_widget(project_id),
            build_total_project_usage_tab_widget(project_id),
            build_total_project_usage_tab_widget(project_id),
        ]
        project_tab_widget.titles = project_tab_widget_titles
        Tabs['projects']['sub_tabs'] = {
            'charge_number': {
                'tab_widget': project_tab_widget,
            },
        }
        with output:
            display(Markdown(f'### {title}'))
            display(project_tab_widget)
    projects_tab_widget.children = projects_tab_children
    projects_tab_widget.titles = projects_tab_titles
    return projects_tab_widget

def build_my_usage_tab_widget(project_id):
    output = widgets.Output()
    tab_widget = widgets.Tab()
    titles = ['Total', 'By Resource']
    tab_widget.children = [
        build_timeseries_tab_with_total(f'person_id = {person_id} and allocation_id = {project_id}'),
        build_my_usage_by_resource_tab(project_id),
    ]
    tab_widget.titles = titles
    with output:
        display(tab_widget)
    return output

def build_timeseries_tab_with_total(where):
    output = widgets.Output()
    df = get_usage_data(where)
    df['Day'] = df['day_id'].apply(lambda x: datetime.strptime(str(x), "%Y00%j").day)
    start = pd.Timestamp(START_DATE)
    end = start + pd.offsets.MonthEnd(0)
    df = (
        df.rename(columns={'total_ace': 'ACCESS Credit Equivalents'})
          .groupby('Day', as_index=False)['ACCESS Credit Equivalents'].sum()
          .set_index('Day')
          .reindex(range(1, end.day + 1), fill_value=0)
          .rename_axis('Day')
    )
    with output:
        display(Markdown(f'Total: {df['ACCESS Credit Equivalents'].sum():,.1f} ACCESS Credit Equivalents'))
        display(Markdown('By Day:'))
        display_timeseries_plot(df)
    return output

def build_my_usage_by_resource_tab(project_id):
    output = widgets.Output()
    tab_widget = widgets.Tab()
    titles = ['Total', 'By Day']
    tab_widget.children = build_aggregate_and_timeseries_tab(
        group_by='resource_id',
        where=f'person_id = {person_id} and allocation_id = {project_id}',
    )
    tab_widget.titles = titles
    with output:
        display(tab_widget)
    return output

def build_total_project_usage_tab_widget(project_id):
    output = widgets.Output()
    tab_widget = widgets.Tab()
    titles = ['Total', 'By User']
    tab_widget.children = [
        build_timeseries_tab_with_total(f'allocation_id = {project_id}'),
        build_my_usage_by_resource_tab(project_id),
    ]
    tab_widget.titles = titles
    with output:
        display(tab_widget)
    return output

def build_aggregate_and_timeseries_tab(group_by, where):
    return [widgets.Output(), widgets.Output()]

def display_timeseries_plot(
    df,
    title=None,
    color=None,
    labels=None,
    category_orders=None,
    color_discrete_map=None,
    vertical_legend=False,
    yaxis_tickformat=',',
    before_show=None,
    caption='',
):
    plot = px.line(
        df,
        y='ACCESS Credit Equivalents',
        title=title,
        color=color,
        labels=labels,
        category_orders=category_orders,
        color_discrete_map=color_discrete_map,
        # By default, for charts with more than 1000 points, Plotly will switch to rendering with WebGL
        # instead of SVG for better performance. However, there is a browser limit to the number of WebGL
        # contexts that can be rendered on the same web page, and there are enough charts with > 1000 points
        # in this notebook to overrun the limit. Thus, we force SVG, which from manual testing does not seem
        # to have a noticeable hit in performance.
        render_mode='svg',
        height=500,
    )
    plot.update_traces(
        hovertemplate='%{y:,.0f}',
    )
    plot.update_layout(
        yaxis_tickformat=yaxis_tickformat,
        legend_title_text='',
        hovermode='x unified',
        hoverlabel_namelength=-1,
    )
    if vertical_legend:
        plot.update_layout(
            legend_orientation='v',
            legend_xanchor='left',
            legend_x=0,
            legend_yanchor='bottom',
            legend_y=-1.3,
        )
    if before_show is not None:
        before_show(plot)
    #figure_widget = go.FigureWidget(plot)
    #section_number = SECTIONS[CURRENT_SECTION_TITLES[0]]['number']
    #filename = f"ACCESS-RP-Report-{RP.replace(' ', '-')}-{START}-{END}-Fig{section_number}_{CURRENT_FIGURE_NUMBER + 1}"
    plot.show(config={
        #'showAxisRangeEntryBoxes': False,
        #'toImageButtonOptions': {'filename': filename},
    })
    #figure_widget.update_layout(width=1000)
    #caption_widget = get_figure_caption(caption)
    return plot

display(build_projects_tab_widget())